### PYSPARK ITERVIEW QUESTIONS

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

###### Q1. While ingesting customer data from an external source, you notice duplicate entires. How would you remove duplicate and retain only the lastest entry based on a timestamp column?

In [0]:
data = [
        ("101","2023-12-01","100"),
        ("101","2023-12-02","150"),
        ("102","2023-12-01","200"),
        ("102","2023-12-02","250")
        ]
columns = ["product_id","date","sales"]
df = spark.createDataFrame(data,columns)
df.display()

###### Solution-1:

In [0]:
## Casting Date Column from string to date format

# Casting Method-1:
# df = df.withColumn('date',to_date(col('date')))
# Casting Method-2:
df = df.withColumn('date',col('date').cast(DateType()))

## Drop Duplicates
df = df.orderBy('product_id','date',ascending = [1,0]).dropDuplicates(subset=['product_id']).display()
 

###### Q2. While processing data from multiple files with inconsistent schemas, you need to merge them into a single DataFrame. How would you handle this inconsistency in PySpark?

In [0]:
## We use the option called, "mergeSchema = True" in order to create a DF on top of all the schemas present in source.

df = spark.read.format('parquet')\
        .option('mergeSchema',True)\
        .load('FileFolder/FileName')

###### Q3. You are working with a real-time data pipeline, and you noticed missing values in your streaming data column - category. How would you handle null or missing values in such a scenario?

###### df_stream = spark.readStream.schema("id INT, value STRING").csv("path/to/stream")

In [0]:
df = df.fillNa({'Category':'N/A'})

## Here we are using dictionary to fill the missing values in the column 'Category' because we are making it generic, in case the requirement is for more than one column, we can simply add the key value pair in this dictionary.

###### Q4. You need to calculate the total number of actions performed by user in a system. How would you calculate the top 5 most active user based on this information?

In [0]:
data = [("user1",5),("user2",8),("user3",2),("user4",10),("user2",3)]
columns = ["user_id","actions"]

df = spark.createDataFrame(data,columns)
df.display()

In [0]:
df = df.groupBy('user_id').agg(sum('actions').alias('total_actions')).orderBy('total_actions',ascending=False).limit(5)
df.display()

###### Q5. While processing sales transaction data, you need to identify the most recent transaction for each customer. How would you approach this task?

In [0]:
data = [("cust1","2023-12-01",100),("cust2","2023-12-02",150),("cust1","2023-12-03",200),("cust2","2023-12-04",250)]
columns = ["customer_id","transaction_date","sales_amount"]

df = spark.createDataFrame(data,columns)
df.display()

In [0]:
## casting date column:
df = df.withColumn("transaction_date",col("transaction_date").cast(DateType()))

## Windows function:
df = df.withColumn("flag",dense_rank().over(Window.partitionBy('customer_id').orderBy(col('transaction_date').desc()))).filter(col('flag')==1).drop('flag')
df.display()

###### Q6. You need to identify customers who haven't made any purchases in the last 30 days. how would you filter such customers?

In [0]:
data = [("Cust1","2025-12-01"),("Cust2","2024-11-20"),("Cust3","2026-09-01"),("Cust4","2023-12-02"),("Cust5","2026-09-21")]
columns = ["customer_id","transaction_date"]

df = spark.createDataFrame(data,columns)
df.display()

In [0]:
df = df.withColumn('transaction_date',to_date('transaction_date'))
df.display()

In [0]:
## Find out the date difference between the current date and the transaction date:

df = df.withColumn('gap',datediff(current_date(),'transaction_date')).filter(col('gap')>30).drop('gap')
df.display()


###### Q7. While analyzing customer reviews, you need to identify the most frequently used words in feedback. How would you implement this?

In [0]:
data = [("customer_1","The product was great"),("customer_2","on time delivery!"),("customer_3","Loved the product, it was great"),("customer_4","The product was amazing"),("customer_5","Not bad!")]
columns = ["customer_id","review"]

df = spark.createDataFrame(data,columns)
df.display()

In [0]:
df = df.withColumn('review',lower('review')).withColumn('review',explode(split('review',' '))).withColumn('review',regexp_replace(col("review"), "[^a-z]", ""))
df = df.groupBy('review').agg(count('review').alias('wordcount')).orderBy('wordcount',ascending=False)
df.display()

In [0]:
df = df.withColumn("dense_rank",dense_rank().over(Window.orderBy(col("wordcount").desc()))).filter(col("dense_rank")==1).drop("dense_rank")
df.display()

###### Q8. You need to calculate the cumulative sum of sales over time for each product. How would you approach this?

In [0]:
data = [("product1","2023-12-01",100),("product2","2023-12-02",150),("product1","2023-12-03",200),("product1","2023-12-04",250),("product2","2023-12-05",300)]
columns = ["product_id","date","sales"]
df = spark.createDataFrame(data,columns)
df.display()


In [0]:
df = df.withColumn("date",to_date("date"))
df = df.withColumn("cum_sum",sum("sales").over(Window.partitionBy("product_id").orderBy(col("date"))))
df.display()

###### Q9. While preparing a data pipeline, you notice some duplicate rows in a dataset. How would you remove the duplicates without affecting the original order?

In [0]:
data = [("John",25),("Jane",30),("John",25),("Alice",22)]
columns = ["name","age"]
df = spark.createDataFrame(data,columns)
df.display()

In [0]:
df = df.withColumn('rank_by_age',row_number().over(Window.partitionBy('name').orderBy('age'))).filter(col('rank_by_age')==1)
df.display()

###### Q10. You are working with user activity data and need to calculate the average session duration per user. How would you implement this? (Added two more cases from my end)

In [0]:
data = [("Ram","2025-12-01",60),("Shyam","2025-12-02",30),("Ram","2025-12-03",20),("Karan","2025-12-03",40),("Ram","2025-12-04",50),("Shyam","2025-12-04",10)]
column = ["user_name","session_date","duration"]

df = spark.createDataFrame(data,column)
df.display()

In [0]:
avg_df = df.groupBy('user_name').agg(avg('duration').alias('average_duration'))
avg_df = avg_df.withColumn('average_duration',round(col('average_duration')))
avg_df.display()

In [0]:
sum_df = df.groupBy('session_date').agg(sum('duration').alias('per_day_total'))
sum_df = sum_df.withColumn('per_day_total',col('per_day_total').cast("decimal(10,2)"))
sum_df.display()

In [0]:
user_df = df.groupBy('user_name').agg(sum('duration').alias('user_total_duration'))
user_df = user_df.withColumn('user_total_duration',col('user_total_duration').cast(DecimalType(10,2))).orderBy('user_total_duration',ascending=False)
user_df.display()

###### Q11. While analyzing sales data, you need to find the product with the highest sales for each month. How would you accomplish this?

In [0]:
data = [("Eggs","2025-12-01","100"),("Apple","2025-12-01","Not Any"),("Banana","2025-11-01","100"),("Eggs","2025-11-02","None"),("Oranges","2025-10-02","100"),("Banana","2025-10-02","500"),("Oil","2025-10-03","10"),("Apple","2025-10-03","250"),("Oranges","2025-11-04","80"),("Banana","2025-11-04","20"),("Eggs","2025-12-04","110"),("Eggs","2025-11-04","50")]
column = ["product","date","quantity"]

df = spark.createDataFrame(data,column)
df.display()

In [0]:

## Replace the other then integer values with 0
null_mapping = {"Not Any":"0", "None":"0"}
df = df.replace(null_mapping, subset=['quantity'])
df.display()

In [0]:
## Cast the quantity and date column, and add new column called month to find the highest sales as per month.
df = df.withColumn("quantity",col("quantity").cast("integer"))
df = df.withColumn("date",col("date").cast("date"))
df = df.withColumn("sale_month",date_format(col("date"),"MMMM"))
df.display()

In [0]:
grp_df = df.groupBy("sale_month","product").agg(sum("quantity").alias("total_sales")).orderBy("sale_month","total_sales",ascending=False)
grp_df.display()

In [0]:
rank_df = grp_df.withColumn("selling_rank",rank().over(Window.partitionBy("sale_month").orderBy(col("total_sales").desc())))
rank_df.display()

In [0]:
final_df = rank_df.filter(col("selling_rank")==1).drop("selling_rank")
final_df.display()

###### Q12. You are working with a large Delta table that is frequently updated by multiple users. The data is stored in partitions and sometimes updates can cause inconsisten reads due to concurrent transactions. How would you ensure ACID compliance and avoid data corruption in PySpark?

In [0]:
df = spark.read.format('parquet').load("source_file_path")

from delta.tables import DeltaTable

delta_tbl = DeltaTable.forPath("source_file_path")

delta_tbl.alias("trg").merge(df.alias("src"),"src.id = trg.id")\
                        .whenNotMatchedInsertAll()\
                        .whenMatchedUpdateAll()\
                        .execute()

###### Q13. You need to process a large dataset stored in PARQUET format and ensure that all columns have the right schema (almost). How would you do this?

In [0]:
df = spark.read.fromat("parquet")\
            .option("inferSchema",True)\
            .load("source_file_path")
## inferSchema is needed to add in order to handle the changes in schema if any

###### Q14. You are reading a CSV file and need to handle corrupt records gracefully by skipping them. How would you configure this in PySpark?

In [0]:
df = spark.read.format("csv")\
            .option("mode","DROPMALFORMED")\
            .load("staging location")

## DROPMALFORMED will drop all the corrupted records.